# **1 CONFIG**

## 1.1 Google Sheets

In [1]:
sheet_link = 'https://docs.google.com/spreadsheets/d/1o_TcGGjIvDj_tTqxBQ-Y9wtPHyKikOF4F3USLaUKeQc/edit#gid=399585107'
sheet_name = 'PR-me5a'
sheet_range = 'A1:X'

result_sheet_name = 'Result'
result_finalization_sheet_name = 'Purchase requisition- finalization'

## 1.2 Parameters

In [2]:
celanese_vendor = 'Celanese Xingda Filaments Company'
celanese_min = 4000

yangzhou_vendor = 'YANGZHOU SANXING PLASTIC CO., LTD'
yangzhou_min = 18500

list_resin_local = ['M38272', 'M43203']

# 2 IMPORT PACKAGE

In [3]:
import pandas as pd
import numpy as np
import datetime

import gspread
import gspread_dataframe as gd

from google.colab import data_table, auth
data_table.enable_dataframe_formatter()
auth.authenticate_user()

from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3 READ AND CLEAN DATA

## 3.1 Read

In [4]:
df_raw = pd.DataFrame.from_records(gc.open_by_url(sheet_link).worksheet(sheet_name).get(sheet_range))
df_raw.columns = df_raw.iloc[0,:]
df_raw = df_raw.iloc[1:,:]
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337 entries, 1 to 337
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Material              337 non-null    object
 1   Short Text            337 non-null    object
 2   Purchase Requisition  337 non-null    object
 3   Quantity Requested    337 non-null    object
 4   Unit of Measure       337 non-null    object
 5   Requisition Date      337 non-null    object
 6   Name of Vendor        337 non-null    object
 7   Deliv. date(From/to)  337 non-null    object
 8   Requisitioner         337 non-null    object
 9   Fixed Vendor          337 non-null    object
 10  Release Date          337 non-null    object
 11  GR Processing Time    337 non-null    object
 12  Bundle                337 non-null    object
 13  Unique/Common         337 non-null    object
 14  Status                337 non-null    object
 15  Lead Time             337 non-null    ob

In [5]:
df_raw.head()

,Material,Short Text,Purchase Requisition,Quantity Requested,Unit of Measure,Requisition Date,Name of Vendor,Deliv. date(From/to),Requisitioner,Fixed Vendor,...,Status,Lead Time,Safety Time,PO approval (day),Min Coverage Day,Requirement Day,Requirement Qty (LT),Stock,PO Pending,Shortage
1,M36803,CLARIANT GREEN DS160344-05 (R) COLORANT,27123658,25.00,KG,11/28/2023,Avient Vietnam Company Limited,12/19/2023,TB RAW PLlan,261799,...,Active,7,17,1,16,1/7/2024,9.3743601,7.552,0,-2
2,M22710,CLARIANT VIOLET DS80219 COLORANT,27126546,100.00,KG,12/4/2023,Avient Vietnam Company Limited,12/19/2023,TB RAW PLlan,261799,...,To be discontinue,7,17,1,16,1/7/2024,245.048896,195.858,0,-49
3,M25062,CLARIANT BLUE DS80556 (R) COLORANT,27126549,225.00,KG,12/4/2023,Avient Vietnam Company Limited,12/19/2023,TB RAW PLlan,261799,...,To be discontinue,7,15,1,14,1/5/2024,214.630656,214.413,0,0
4,M32455,CLARIANT SPARKLE PINK 102969-13 (R),27126559,75.00,KG,12/4/2023,Avient Vietnam Company Limited,12/19/2023,TB RAW PLlan,261799,...,To be discontinue,7,17,1,16,1/7/2024,248.507072,184.675,0,-64
5,M28303,CLARIANT GREEN DS100038-05 (R) COLORANT,27127534,25.00,KG,12/6/2023,Avient Vietnam Company Limited,12/19/2023,TB RAW PLlan,261799,...,Active,7,16,1,15,1/6/2024,35.784144,41.639,0,6


## 3.2 Clean

In [6]:
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.astype(str)
df_raw = df_raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df_raw = df_raw.replace({'#N/A': None, '#VALUE!': None, '#REF!': None, 'None': None, '': None})
df_raw = df_raw[~df_raw['Material'].isnull()]
df_raw = df_raw[~df_raw['Bundle'].isnull()]

In [7]:
numeric_columns = ['Quantity Requested', 'GR Processing Time', 'Lead Time', 'Safety Time', 'Min Coverage Day', 'Requirement Qty (LT)', 'Stock', 'PO Pending', 'Shortage']

# Replace characters in the specified columns
df_raw[numeric_columns] = df_raw[numeric_columns].replace({',': '', '\$': '', r'\(': '', r'\)': ''}, regex=True)
df_raw[numeric_columns] = df_raw[numeric_columns].apply(pd.to_numeric, errors='coerce')

In [8]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 288 entries, 1 to 288
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Material              288 non-null    object 
 1   Short Text            288 non-null    object 
 2   Purchase Requisition  288 non-null    object 
 3   Quantity Requested    288 non-null    float64
 4   Unit of Measure       288 non-null    object 
 5   Requisition Date      288 non-null    object 
 6   Name of Vendor        286 non-null    object 
 7   Deliv. date(From/to)  288 non-null    object 
 8   Requisitioner         288 non-null    object 
 9   Fixed Vendor          286 non-null    object 
 10  Release Date          288 non-null    object 
 11  GR Processing Time    288 non-null    int64  
 12  Bundle                288 non-null    object 
 13  Unique/Common         288 non-null    object 
 14  Status                288 non-null    object 
 15  Lead Time             2

In [9]:
df = df_raw

# 4 CHOOSE PR

## 4.1 CALCULATE CUM SUM TO MAKE DECISION

In [10]:
df['Deliv. date(From/to)'] = pd.to_datetime(df['Deliv. date(From/to)'], format='%m/%d/%Y')

In [11]:
df.head(1)

,Material,Short Text,Purchase Requisition,Quantity Requested,Unit of Measure,Requisition Date,Name of Vendor,Deliv. date(From/to),Requisitioner,Fixed Vendor,...,Status,Lead Time,Safety Time,PO approval (day),Min Coverage Day,Requirement Day,Requirement Qty (LT),Stock,PO Pending,Shortage
1,M36803,CLARIANT GREEN DS160344-05 (R) COLORANT,27123658,25.0,KG,11/28/2023,Avient Vietnam Company Limited,2023-12-19,TB RAW PLlan,261799,...,Active,7,17,1,16,1/7/2024,9.37436,7.552,0,-2


In [12]:
# Sort by 'Name of Vendor' and then by 'Deliv. date(From/to)' and 'Quantity Requested'
df = df.sort_values(by=['Name of Vendor', 'Material', 'Deliv. date(From/to)', 'Quantity Requested'], ascending=[True, True, True, True])

# Reset index to make it cleaner if needed
df.reset_index(drop=True, inplace=True)

In [13]:
# Create column 'Cum Sum'
df['Cum Sum'] = df.groupby('Material')['Quantity Requested'].cumsum()

In [14]:
# Create column 'Option' to choose PRs so that they could cover Requirement
df['Option'] = np.where((df['Shortage'] + df['Cum Sum']) < df['Quantity Requested'], 1, 0)

## 4.2 NOTE

### Chú ý thứ tự các trường hợp trong column Note cho hợp lý

In [15]:
# Create column 'Note'
df['Note'] = 'OK'

In [16]:
# Thiếu PR
df.loc[(df.groupby('Material')['Cum Sum'].transform('max') + df.groupby('Material')['Shortage'].transform('min')) < 0, 'Note'] = 'Thiếu PR'

In [17]:
df['Exeption Supplier Note'] = pd.NA

In [18]:
# Celanese
# Rule của Celanese là phải mua >= 4000
celanese_total = df.loc[df['Name of Vendor'] == celanese_vendor, 'Quantity Requested'].mul(df['Option']).sum()

if celanese_total == 0 or celanese_total >= celanese_min:
    df.loc[df['Name of Vendor'] == celanese_vendor, 'Exeption Supplier Note'] = 'OK'
else:
    df.loc[df['Name of Vendor'] == celanese_vendor, 'Exeption Supplier Note'] = f'Under min, shortage {(celanese_min - celanese_total):.0f} kg'

In [19]:
# Yangzhou
# Rule của Yangzhou là phải mua chia hết cho 18500
yangzhou_total = df.loc[df['Name of Vendor'] == yangzhou_vendor, 'Quantity Requested'].mul(df['Option']).sum()

if (yangzhou_total // yangzhou_min) == 0:
    df.loc[df['Name of Vendor'] == yangzhou_vendor, 'Exeption Supplier Note'] = 'OK'
else:
    df.loc[df['Name of Vendor'] == yangzhou_vendor, 'Exeption Supplier Note'] = f'Shortage {yangzhou_min - (yangzhou_total // yangzhou_min):.0f} kg or Excess {(yangzhou_total // yangzhou_min):.0f} kg'

In [20]:
# Local M
df.loc[df['Material'].isin(list_resin_local), 'Option'] = 0
df.loc[df['Material'].isin(list_resin_local), 'Note'] = 'Local Resin'

In [21]:
# Null Vender
df.loc[df['Name of Vendor'].isnull(), 'Option'] = 0
df.loc[df['Name of Vendor'].isnull(), 'Note'] = 'Block Source List'

In [22]:
# Discontinue M
df.loc[(df['Status'] == 'To be discontinued') | (df['Status'] == 'New'), 'Note'] = 'MRP Check'

In [23]:
df

,Material,Short Text,Purchase Requisition,Quantity Requested,Unit of Measure,Requisition Date,Name of Vendor,Deliv. date(From/to),Requisitioner,Fixed Vendor,...,Min Coverage Day,Requirement Day,Requirement Qty (LT),Stock,PO Pending,Shortage,Cum Sum,Option,Note,Exeption Supplier Note
0,P11270106,ELASTIC STAPLE-HANGER-150MM-10KPCS/ROLL,27109760,40.0,ROL,11/1/2023,AVERY DENNISON (HK) LTD,2024-02-03,Import Pack,8147554,...,28,3/18/2024,495.213919,253.189,280,38,40.0,0,OK,NaN
1,M28931,POLYONE GREEN 4E17688 COLORANT,27123651,1000.0,KG,11/28/2023,"Avient (THAILAND) CO., LTD.",2024-01-08,TB RAW PLlan,210504,...,28,2/15/2024,2262.438787,1237.047,1000,-25,1000.0,1,OK,NaN
2,M22710,CLARIANT VIOLET DS80219 COLORANT,27126546,100.0,KG,12/4/2023,Avient Vietnam Company Limited,2023-12-19,TB RAW PLlan,261799,...,16,1/7/2024,245.048896,195.858,0,-49,100.0,1,OK,NaN
3,M25044,REMAFIN PURPLE DS80390-05 (R ),27130018,775.0,KG,12/11/2023,Avient Vietnam Company Limited,2023-12-23,TB RAW PLlan,261799,...,16,1/11/2024,928.593150,990.816,0,62,775.0,0,OK,NaN
4,M25062,CLARIANT BLUE DS80556 (R) COLORANT,27126549,225.0,KG,12/4/2023,Avient Vietnam Company Limited,2023-12-19,TB RAW PLlan,261799,...,14,1/5/2024,214.630656,214.413,0,0,225.0,0,OK,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,M41908,ZHEJIANG REEF PCR PPH EFP413,27113072,25500.0,KG,11/7/2023,"Zhejiang Lifu Technology Co., Ltd.",2024-01-22,TB RAW PLlan,8259466,...,36,3/31/2024,187121.348500,50083.961,127500,-9537,25500.0,1,OK,NaN
284,M41908,ZHEJIANG REEF PCR PPH EFP413,27127833,25500.0,KG,12/6/2023,"Zhejiang Lifu Technology Co., Ltd.",2024-01-22,TB RAW PLlan,8259466,...,36,3/31/2024,187121.348500,50083.961,127500,-9537,51000.0,0,OK,NaN
285,M41908,ZHEJIANG REEF PCR PPH EFP413,27130674,25500.0,KG,12/12/2023,"Zhejiang Lifu Technology Co., Ltd.",2024-02-01,TB RAW PLlan,8259466,...,36,3/31/2024,187121.348500,50083.961,127500,-9537,76500.0,0,OK,NaN
286,M43749,BBC 610 SPIRAL WT/BL502+WT609 7.5MIL*33,27127548,100.0,KG,12/6/2023,None,2024-02-19,TB RAW PLlan,None,...,28,4/8/2024,335.549190,281.977,0,-54,100.0,0,Block Source List,NaN


# 5 WRITE RESULT

In [24]:
# Format column 'Deliv. date(From/to)' back to string m/d/Y
df['Deliv. date(From/to)'] = pd.to_datetime(df['Deliv. date(From/to)'], format='%m/%d/%Y').dt.strftime('%m/%d/%Y')

In [25]:
# Tất cả kết quả
result_link = sheet_link
result_wb = gc.open_by_url(result_link)
result_wb.worksheet(result_sheet_name).clear()
gd.set_with_dataframe(result_wb.worksheet(result_sheet_name),df)

In [27]:
# Kết quả lấy filter chọn 1 hoặc MRP check
df_finalization = df[(df['Option'] == 1) | (df['Note'] == 'MRP Check')]

result_finalization_link = sheet_link
result_finalization_wb = gc.open_by_url(result_finalization_link)
result_finalization_wb.worksheet(result_finalization_sheet_name).clear()
gd.set_with_dataframe(result_finalization_wb.worksheet(result_finalization_sheet_name),df_finalization)

In [28]:
print('Done')

Done
